# 继承与对象协议

学习目标：能用继承或组合组织对象协作，并通过方法约定让自定义对象参与常用 Python 操作。

前置知识：类与实例、self、实例属性、初始化方法、方法调用、函数参数、容器操作、真值判断和基本 match 语句。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 继承、重写与组合

### 1.1 从已有类继承行为

继承（inheritance）让新类沿用已有类的属性和方法。被继承的类称为基类，也叫父类；新类称为派生类，也叫子类。下面的 Lamp 是 Device 的子类。

子类没有定义某个方法时，可以沿继承关系查找它。这里 Lamp 没有定义初始化方法，创建实例时使用 Device 的 \_\_init\_\_；name 仍然是新实例自己的属性。

In [1]:
class Device:
    """保存设备名称并提供通用状态描述。"""

    def __init__(self, name):
        self.name = name

    def status(self):
        """返回设备当前的状态文字。"""
        return f"{self.name}：待机"

    def report(self):
        """通过当前对象的状态方法生成报告。"""
        return f"[{self.status()}]"


class Lamp(Device):
    """沿用通用设备行为的灯。"""


lamp = Lamp("台灯")
print(lamp.name)  # 台灯；初始化方法来自 Device。
print(lamp.report())  # [台灯：待机]；两个方法都从 Device 继承。

台灯
[台灯：待机]


### 1.2 用重写改变同一操作

重写（overriding）是在子类中定义与基类同名的方法，为子类提供自己的行为。调用对象的方法时，会使用适用于这个对象的实现。

基类方法内部通过 self 调用另一个方法时，也会受到重写影响。下面继续使用 Device；即使 report 来自 Device，它调用的 status 仍可能来自子类。

In [2]:
class AlertLamp(Device):
    """用状态文字提示关注的灯。"""

    def status(self):
        """返回需要关注的状态。"""
        return f"{self.name}：请检查"


normal = Device("电源")
alert = AlertLamp("信号灯")
print(normal.report())  # [电源：待机]
print(alert.status())  # 信号灯：请检查
print(alert.report())  # [信号灯：请检查]；继承的方法调用了重写的方法。

[电源：待机]
信号灯：请检查
[信号灯：请检查]


### 1.3 用组合连接不同职责

组合（composition）让一个对象把其他对象保存在属性中，再调用这些对象完成工作；把操作交给所持有对象的做法称为委托（delegation）。它不要求两者有继承关系。

在这个例子中，警示灯是一种设备，而书桌拥有一盏灯，所以用继承表达前者，用组合表达后者。两种做法可以一起使用：Desk 持有的恰好是 Device 的子类实例。

In [3]:
class Desk:
    """持有一盏灯，并使用它提供照明状态。"""

    def __init__(self, lamp):
        self.lamp = lamp

    def describe(self):
        """把状态查询交给持有的灯对象。"""
        return f"书桌 / {self.lamp.report()}"


# 复用前面定义的 Lamp 和 AlertLamp。
desk = Desk(Lamp("阅读灯"))
print(desk.describe())  # 书桌 / [阅读灯：待机]
desk.lamp = AlertLamp("阅读灯")
print(desk.describe())  # 书桌 / [阅读灯：请检查]；替换的是组成对象。

书桌 / [阅读灯：待机]
书桌 / [阅读灯：请检查]


## 2 多态、鸭子类型与类型检查

### 2.1 对相同接口使用不同对象

多态（polymorphism）体现在同一段调用代码可以使用不同对象，由各对象提供相应行为。鸭子类型（duck typing）是一种按对象支持的操作来使用它的编程风格，不要求先检查具体类型。

鸭子类型可以实现多态，而多态并不要求所有对象继承同一个业务基类。下面的调用方只要求 report() 不接收额外实参并返回字符串；仅有同名方法、却不满足这个约定，也不能正确协作。

In [4]:
class StatusCard:
    """独立保存一段状态文字，不继承 Device。"""

    def __init__(self, text):
        self.text = text

    def report(self):
        """按调用方约定返回状态字符串。"""
        return f"[卡片：{self.text}]"


# Device、AlertLamp 来自前文；循环中的调用方式保持一致。
sources = [Device("电源"), AlertLamp("信号灯"), StatusCard("已检查")]
for source in sources:
    print(source.report())
# 依次输出 [电源：待机]、[信号灯：请检查]、[卡片：已检查]。

[电源：待机]
[信号灯：请检查]
[卡片：已检查]


### 2.2 需要判断类型关系时再检查

isinstance 检查对象是否属于某个类或其子类；issubclass 检查类之间的关系，并把一个类视为它自己的子类。下面的 device 是待检查的对象，Device 和 AlertLamp 是类。

| 名称 | 中文名称／含义 | 本例写法 |
| --- | --- | --- |
| isinstance | 实例类型检查，接受子类实例 | isinstance(device, Device) |
| issubclass | 类的子类关系检查 | issubclass(AlertLamp, Device) |
| type | 获取对象的实际类型 | type(device) |

type(device) is Device 只检查实际类型是否恰好为 Device。类型关系检查和鸭子类型回答的问题不同：前者确认归属，后者关注当前操作能否完成。ABC 还支持虚拟子类等扩展识别方式，不能把所有 isinstance 结果都解释为实际继承；本章示例采用普通继承。

In [5]:
# 复用本章的 Device、AlertLamp 和 StatusCard。
device = AlertLamp("信号灯")
print(isinstance(device, Device))  # True；子类实例也属于基类。
print(type(device) is Device)  # False；实际类型是 AlertLamp。
print(issubclass(AlertLamp, Device))  # True
print(issubclass(Device, Device))  # True
print(isinstance(StatusCard("就绪"), Device))  # False；但仍能调用 report。
print(issubclass(bool, int))  # True；bool 也是 int 的子类。

True
False
True
True
False
True


## 3 super 与协作式方法解析顺序

### 3.1 保留已有实现，再增加行为

方法解析顺序（method resolution order，MRO）规定沿类层次查找方法的先后次序。super() 返回一个代理，用它继续查找当前类之后的实现。

在普通实例方法中，无参数 super() 根据定义该方法的类和 self 确定查找起点。下面是单继承，所以恰好找到直接基类；这个现象不能推广成“super 总是调用直接父类”。

这里重写 \_\_init\_\_ 后，用 super().\_\_init\_\_(name) 明确完成已有初始化，再添加位置属性；name 表示设备名称。

In [6]:
class LocatedLamp(AlertLamp):
    """在警示灯行为上增加摆放位置。"""

    def __init__(self, name, place):
        super().__init__(name)
        self.place = place

    def status(self):
        """保留已有状态，再添加位置信息。"""
        return f"{super().status()}，位置：{self.place}"


located = LocatedLamp("信号灯", "门口")
print(located.name)  # 信号灯；已有初始化沿继承关系完成。
print(located.report())  # [信号灯：请检查，位置：门口]

信号灯
[信号灯：请检查，位置：门口]


### 3.2 菱形继承中，下一站可能是兄弟类

Left 和 Right 都继承 Root，Branch 再继承 Left、Right，这种两条路径汇合到同一基类的关系叫菱形继承。类的 \_\_mro\_\_ 属性保存实际查找顺序；示例中 base 是遍历到的类，\_\_name\_\_ 是它的名称。

Python 使用 C3 方法解析顺序，使查找顺序满足继承关系与基类次序的约束，共同祖先只在 MRO 中出现一次。省略基类的普通类最终继承 object。这里直接观察结果，不手工计算 C3。

协作调用还需要方法本身配合：各层采用兼容的参数约定，需要继续时调用一次 super，末端负责结束。MRO 不会自动执行所有方法。下面 Root.steps 是终点，因此不再继续查找。

In [7]:
class Root:
    """提供步骤收集的终点。"""

    def steps(self):
        """返回终点名称。"""
        return ["Root"]


class Left(Root):
    """收集左侧步骤后继续协作。"""

    def steps(self):
        """把后续步骤追加在左侧步骤之后。"""
        # 对 Branch 实例，super 沿 Branch 的 MRO 前进，下一站是 Right。
        return ["Left"] + super().steps()


class Right(Root):
    """收集右侧步骤后继续协作。"""

    def steps(self):
        """把后续步骤追加在右侧步骤之后。"""
        return ["Right"] + super().steps()


class Branch(Left, Right):
    """把两个分支合并为一次协作调用。"""

    def steps(self):
        """从汇合类开始收集步骤。"""
        return ["Branch"] + super().steps()


print([base.__name__ for base in Branch.__mro__])
# ['Branch', 'Left', 'Right', 'Root', 'object']
print(Branch().steps())
# ['Branch', 'Left', 'Right', 'Root']；Left 的下一站是 Right。
print(Left().steps())  # ['Left', 'Root']；实例类型不同，后续顺序也不同。

['Branch', 'Left', 'Right', 'Root', 'object']
['Branch', 'Left', 'Right', 'Root']
['Left', 'Root']


### 3.3 调整基类顺序后的协作

继续使用前面的 Root、Left 和 Right，把基类次序交换后，协作顺序也随之变化。硬编码调用 Root.steps(self) 会绕开 MRO 中本应继续处理的其他实现。

双参数 super(Left, branch) 中，Left 指定从哪个类之后开始查找，branch 是决定所用 MRO 的实例。它不表示创建一个 Left 对象，也不表示调用 Left 自己的方法。

复杂继承如果提出互相矛盾的次序约束，类定义会因无法得到一致 MRO 而引发 TypeError；并非任意基类排列都可组合。

In [8]:
class Reverse(Right, Left):
    """交换两个分支的优先次序。"""


print([base.__name__ for base in Reverse.__mro__])
# ['Reverse', 'Right', 'Left', 'Root', 'object']
print(Reverse().steps())  # ['Right', 'Left', 'Root']

branch = Branch()
print(super(Left, branch).steps())  # ['Right', 'Root']；从 Left 之后开始。

['Reverse', 'Right', 'Left', 'Root', 'object']
['Right', 'Left', 'Root']
['Right', 'Root']


## 4 用抽象基类约定必须提供的方法

### 4.1 标记尚未完成的接口

抽象基类（abstract base class，ABC）可以声明子类必须提供的方法。继承 abc.ABC 后，用 @abc.abstractmethod 标记抽象方法；仍有未实现抽象方法的类不能实例化。

| 名称 | 中文名称／含义 |
| --- | --- |
| abc.ABC | 创建抽象基类时使用的辅助基类 |
| abc.abstractmethod | 标记抽象方法的装饰器 |

@ 写在 def 前，是装饰器语法：定义方法时把函数交给指定装饰器处理，这里用于添加抽象标记。完整的装饰器用法留到“装饰器”专题。禁止实例化的原因是抽象标记，不是方法体中的 pass。

下面用 try 执行预期失败的实例化，用 except TypeError 只捕获“当前类不能这样实例化”的异常并显示其类型；异常处理机制将在后续专题展开。

In [9]:
import abc


class Formatter(abc.ABC):
    """要求子类提供文字格式化行为。"""

    @abc.abstractmethod
    def format_text(self, text):
        """把输入字符串转换为供展示的字符串。"""
        pass

    def render(self, text):
        """在格式化结果外加上统一边框。"""
        return f"[{self.format_text(text)}]"


class MissingFormatter(Formatter):
    """尚未补齐抽象方法的子类。"""


for formatter_type in (Formatter, MissingFormatter):
    try:
        formatter_type()
    except TypeError as error:
        print(type(error).__name__, formatter_type.__name__)
# 依次输出 TypeError Formatter、TypeError MissingFormatter。

TypeError Formatter
TypeError MissingFormatter


### 4.2 补齐方法后使用共同逻辑

继续使用 Formatter，实现全部抽象方法的子类就可以实例化，也可以继承抽象基类中的普通方法。ABC 也允许抽象方法带有可供 super 调用的实现；方法体是否为空与抽象标记是两回事。

ABC 为接口约定增加了显式声明和实例化限制，是对鸭子类型的补充。它不会自动核验实现的业务含义或参数、返回值是否符合约定；这里仍需保证 format\_text 接收字符串并返回字符串。

In [10]:
class UpperFormatter(Formatter):
    """提供具体的大写转换行为。"""

    def format_text(self, text):
        """返回输入文字的大写形式。"""
        return text.upper()


formatter = UpperFormatter()
print(formatter.render("python"))  # [PYTHON]；render 由基类提供。
print(isinstance(formatter, Formatter))  # True

[PYTHON]
True


## 5 对象协议与文字表示

本章的协议（protocol）指对象支持某项操作时遵循的方法及行为约定。特殊方法（special method）是连接自定义类与 Python 内置操作的入口，通常使用两端各有双下划线的名称，也常称为 dunder 方法。

“对象协议”讨论运行时操作约定；后续静态类型专题的 typing.Protocol 则把接口要求提供给类型检查器，两者有关联，但不需要先继承 typing.Protocol 才能使用本章协议。

| 名称 | 中文名称／含义 | 常见入口 |
| --- | --- | --- |
| \_\_repr\_\_ | 便于调试、尽量明确的字符串表示 | repr(book) |
| \_\_str\_\_ | 适合阅读的字符串表示 | str(book)、print(book) |

book 表示下例的 Book 实例。两个方法都必须返回字符串；repr 的形式应尽可能有助于理解或重建对象，但不是任何表示都能当代码执行。默认 \_\_str\_\_ 会使用 \_\_repr\_\_。

特殊方法要定义在类上，不能只给某个实例添加同名属性就期待内置操作调用它。特殊方法查找的完整机制留到“描述器与元类”专题。

In [11]:
class Book:
    """保存一本书的标题，并提供两种文字表示。"""

    def __init__(self, title):
        self.title = title

    def __repr__(self):
        # !r 在 f-string 中使用标题的 repr，保留字符串引号。
        return f"Book({self.title!r})"

    def __str__(self):
        return f"《{self.title}》"


class Label:
    """只定制调试表示的简短标签。"""

    def __init__(self, text):
        self.text = text

    def __repr__(self):
        return f"Label({self.text!r})"


book = Book("Python 入门")
print(repr(book))  # Book('Python 入门')
print(book)  # 《Python 入门》
print(str(Label("待办")))  # Label('待办')；默认 str 使用 repr。

Book('Python 入门')


《Python 入门》
Label('待办')


## 6 长度与真值

### 6.1 让 len 获取元素数量

\_\_len\_\_ 返回对象长度，必须是非负整数。下面用实例属性保存条目列表，把长度查询委托给这个列表。

对象没有定义 \_\_bool\_\_ 时，真值判断会使用 \_\_len\_\_：长度为零是假，非零是真；两个方法都没有时，普通对象默认是真。长度与真值因此有关联，但不是同一项协议。

In [12]:
class Bag:
    """保存一组条目，并报告条目数量。"""

    def __init__(self, items):
        self.items = list(items)

    def __len__(self):
        return len(self.items)


empty_bag = Bag([])
full_bag = Bag(["笔", "本"])
print(len(empty_bag), bool(empty_bag))  # 0 False
print(len(full_bag), bool(full_bag))  # 2 True
print(bool(object()))  # True；object 没有定制长度或真值方法。

0 False
2 True
True


### 6.2 让 bool 表达明确的状态

\_\_bool\_\_ 必须返回 True 或 False，定义后真值判断优先使用它，不再按长度直接判断。下面继续使用 Bag，把“已就绪且装有条目”作为 ReadyBag 的真值含义，ready 参数使用布尔值。

这种设计需要明确告知调用者：非空和可使用可能不同。若只是表达容器是否为空，通常沿用长度对应的真值即可。

In [13]:
class ReadyBag(Bag):
    """只有就绪且非空时才视为可使用的包。"""

    def __init__(self, items, ready):
        super().__init__(items)
        self.ready = ready

    def __bool__(self):
        return bool(self.ready) and bool(self.items)


bag = ReadyBag(["笔"], False)
print(len(bag), bool(bag))  # 1 False；非空但尚未就绪。
bag.ready = True
print(bool(bag))  # True；就绪且非空。
print(bool(ReadyBag([], True)))  # False；就绪但没有条目。

1 False
True
False


## 7 索引、成员检测与迭代

### 7.1 用索引读取对象内容

\_\_getitem\_\_ 支持方括号读取。下面的 shelf 是书架实例，shelf[0] 读取索引为 0 的条目；方法参数 index 接收索引，使用切片时则接收 slice 对象。

示例把操作转交给列表，所以支持负数索引和切片。负数索引与切片并不是定义这个方法后自动获得的，是否支持由实现决定。序列索引越界应引发 IndexError，映射缺键则应引发 KeyError。

序列与映射都可以使用这个方法，因此“能用方括号读取”不足以说明对象是序列。这里按整数位置读取，模拟的是序列读取行为。

In [14]:
class Shelf:
    """按位置读取书名，把读取规则交给内部列表。"""

    def __init__(self, titles):
        self.titles = list(titles)

    def __getitem__(self, index):
        return self.titles[index]


shelf = Shelf(["语法", "函数", "对象"])
print(shelf[0])  # 语法
print(shelf[-1])  # 对象；负数索引由列表处理。
print(shelf[1:])  # ['函数', '对象']；切片返回列表，不是 Shelf。
# 只有三个条目，读取 shelf[3] 会引发 IndexError，此处不执行。

语法
对象
['函数', '对象']


### 7.2 明确成员检测的含义

\_\_contains\_\_ 为 in 和 not in 提供成员检测。下面的 catalog 表示目录实例，"P1" in catalog 查询编号是否存在；方法参数 code 表示待查询的编号。

对于映射语义，成员检测应查键。本例保存编号与书名的对应关系，所以检查编号，而不是书名。实现成员检测不要求同时实现索引或迭代。

In [15]:
class Catalog:
    """保存编号到书名的对应关系，按编号检测成员。"""

    def __init__(self, titles):
        self.titles = dict(titles)

    def __contains__(self, code):
        return code in self.titles


catalog = Catalog({"P1": "语法", "P2": "函数"})
print("P1" in catalog)  # True；编号存在。
print("语法" in catalog)  # False；书名是值，不是被检查的键。
print("P3" not in catalog)  # True

True
False
True


### 7.3 让已有容器负责遍历

\_\_iter\_\_ 返回迭代器（iterator），即负责逐项提供内容的对象。可迭代对象（iterable）能够提供迭代器；两者不是同一个概念。本例每次调用 iter(self.titles)，都从内部列表获得新的迭代器，而不是直接返回列表本身。

有些索引对象也可以通过从 0 开始、以 IndexError 结束的旧式序列协议被遍历；这里显式定义 \_\_iter\_\_，不依赖这种回退。迭代器耗尽、自定义迭代器与 yield 留到“迭代器与生成器”专题。

未定义 \_\_contains\_\_ 时，成员检测会先尝试迭代，再尝试旧式序列协议。因此可迭代对象也可能支持 in，尽管没有专门的成员检测方法。

In [16]:
class ReadingList:
    """提供可以反复遍历的阅读条目。"""

    def __init__(self, titles):
        self.titles = list(titles)

    def __iter__(self):
        return iter(self.titles)


reading = ReadingList(["语法", "函数"])
for title in reading:
    print(title)  # 依次输出 语法、函数。
print(list(reading))  # ['语法', '函数']；新一轮遍历从头开始。
print(list(reading))  # ['语法', '函数']；不会共用已耗尽的迭代器。
print("函数" in reading)  # True；成员检测使用迭代回退。

语法
函数
['语法', '函数']
['语法', '函数']
True


## 8 相等、NotImplemented 与哈希

### 8.1 为对象定义相等规则

\_\_eq\_\_ 参与 == 比较。下面 self 是当前 Product 实例，other 是另一个操作数；本例仅为实际类型相同的对象定义按编号比较，是否跨类型比较需要单独设计。

| 名称 | 中文名称／含义 |
| --- | --- |
| \_\_eq\_\_ | 相等比较方法 |
| NotImplemented | 当前方法不支持这组操作数的特殊返回值 |

无法处理某种类型时返回 NotImplemented，让 Python 有机会采用另一侧的比较方法或回退规则。它与 False 不同：False 表示已经得出“不相等”的结果。

身份检查 is 不受 \_\_eq\_\_ 定义影响。两个不同实例可以按值相等，却仍不是同一个对象。

In [17]:
class Product:
    """用可以修改的编号表达产品的值。"""

    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        if type(self) is not type(other):
            return NotImplemented
        return self.code == other.code


first = Product("P1")
second = Product("P1")
print(first == second)  # True；比较的是编号。
print(first is second)  # False；它们是不同实例。
print(first == Product("P2"))  # False
print(first.__eq__("P1") is NotImplemented)  # True；直接观察特殊返回值。
print(first == "P1")  # False；双方未支持比较，== 最终按身份回退。

True
False
False
True
False


### 8.2 给另一侧一次比较机会

继续使用 Product。下面 ProductCode 为编号提供另一种表示，并明确接受 Product 或 ProductCode 作为比较对象；产品侧先返回 NotImplemented，Python 可以让编号侧继续比较。

若右侧类型是左侧类型的严格子类，右侧的对应比较方法还有优先机会，所以不能把所有 == 都说成“固定先调用左侧方法”。当两侧都不支持时，== 回退到 is，!= 回退到 is not；其他运算不一定有这种回退。

| 名称 | 中文名称／含义 | 用法边界 |
| --- | --- | --- |
| NotImplemented | 不支持当前操作数的返回值 | 在规定的运算特殊方法中返回 |
| NotImplementedError | 表示所需实现尚未完成的异常类型 | 用 raise 引发异常 |

两者不能互换；NotImplemented 也不应拿去做 bool 判断。NotImplementedError 不会启动比较回退，异常的 raise 语法留到后续专题。

In [18]:
class ProductCode:
    """允许与产品对象互相比对的编号表示。"""

    def __init__(self, code):
        self.code = code

    def __eq__(self, other):
        if isinstance(other, (Product, ProductCode)):
            return self.code == other.code
        return NotImplemented


product = Product("P1")
code = ProductCode("P1")
print(product.__eq__(code) is NotImplemented)  # True；产品侧不接受该类型。
print(product == code)  # True；随后由 ProductCode 的方法完成比较。
print(code == product)  # True；编号侧可以直接处理 Product。
print(code == ProductCode("P2"))  # False

True
True
True
False


### 8.3 相等与哈希必须保持一致

可哈希对象能够参与 dict 键和 set 元素的查找。\_\_hash\_\_ 由 hash() 使用并返回整数；对象的哈希值在其生命周期内必须保持不变。

设 a、b 为两个可哈希对象，必须满足：若 a == b，则 hash(a) == hash(b)。反过来不成立，同一个哈希值不代表相等，哈希不能取代相等比较。

若需要自定义值对象的哈希，应根据参与相等比较的字段计算，并保证相关状态稳定。当前先用由字符串组成的元组作为键，避免提前实现不可变类。

In [19]:
first_key = ("P1", "中文")
second_key = tuple(["P1", "中文"])

print(first_key == second_key)  # True
print(hash(first_key) == hash(second_key))  # True；相等键的哈希必须相同。

stock = {first_key: 3}
print(stock[second_key])  # 3；相等的键可以查到同一条目。
print(hash(1) == hash(1.0))  # True；数值相等也要满足相同哈希的约定。
# 不固定字符串相关哈希的具体整数；不同 Python 进程中可能不同。

True
True
3
True


### 8.4 可变值对象保持不可哈希

一个类定义了 \_\_eq\_\_、却没有定义 \_\_hash\_\_ 时，Python 会自动把它的 \_\_hash\_\_ 设为 None，使实例不可哈希。前面的 Product 正是如此；hash() 或把实例用作字典键会引发 TypeError。

Product 的编号可以改变，又参与相等比较，所以不应为它增加基于该编号的哈希方法，也不应直接恢复 object 的身份哈希来搭配值相等。需要做键时，可提取当时的稳定字段值组成元组。

可变性与可哈希性有关，但不是同义词：按身份比较的普通用户类实例默认可哈希；这里讨论的是按可变字段比较的值对象。

In [20]:
# 复用前面的 Product；只捕获这次预期的不可哈希异常。
product = Product("P1")
print(Product.__hash__ is None)  # True；由定义 __eq__ 的规则自动设置。

True

In [21]:
# 预期 TypeError：直接观察原始异常，之后继续运行下一单元。
# TypeError
hash(product)

TypeError: unhashable type: 'Product'

In [22]:
saved_key = (product.code,)
stock = {saved_key: 3}
product.code = "P2"
print(product == Product("P1"))  # False；参与相等比较的编号已改变。
print(saved_key, stock[saved_key])  # ('P1',) 3；元组保留原来的字符串值。

False
('P1',) 3


## 9 用类模式读取对象属性

### 9.1 通过关键字模式检查属性

类模式先用 isinstance 的规则检查对象是否属于指定类，再匹配属性。它与鸭子类型不同：只拥有同名属性的无关类型不会因此匹配这个类模式。

下面 Point 表示平面点，x、y 是它的两个坐标属性。case Point(x=0, y=height) 中，0 是要匹配的字面值，height 是捕获 y 属性值的名称；这种写法无需 \_\_match\_args\_\_。

case 中的 Point(...) 是匹配已有对象的模式，不是创建新 Point 的调用。

In [23]:
class Point:
    """保存平面点的两个坐标。"""

    def __init__(self, x, y):
        self.x = x
        self.y = y


def describe_point(point):
    """按点的位置返回描述，不修改输入对象。"""
    match point:
        case Point(x=0, y=height):
            return f"纵轴上，高度 {height}"
        case Point(x=horizontal, y=vertical):
            return f"一般点：{horizontal}, {vertical}"
        case _:
            return "不是 Point 实例"


print(describe_point(Point(0, 3)))  # 纵轴上，高度 3
print(describe_point(Point(2, 3)))  # 一般点：2, 3
print(describe_point((0, 3)))  # 不是 Point 实例；元组不会匹配 Point。

纵轴上，高度 3
一般点：2, 3
不是 Point 实例


### 9.2 用 \_\_match\_args\_\_ 约定位置模式

\_\_match\_args\_\_ 是由属性名字符串组成的元组，按次序把位置模式转换成属性模式。它是类属性，不是方法，也不按 \_\_init\_\_ 的参数顺序自动推断。

下面 MatchPoint 继承 Point，声明第一个位置对应 x，第二个位置对应 y。因此 MatchPoint(0, height) 与 MatchPoint(x=0, y=height) 的属性约束相同。位置数量超过声明长度会引发 TypeError，而不是普通的匹配失败。

类模式既能使用继承关系识别子类，也能通过属性约定提取值；这两步各有作用。

In [24]:
class MatchPoint(Point):
    """为平面点增加位置模式的属性次序约定。"""

    __match_args__ = ("x", "y")


point = MatchPoint(0, 4)
match point:
    case MatchPoint(0, height):
        print("纵轴高度", height)  # 纵轴高度 4
    case _:
        print("不在纵轴上")  # 本例不输出；point 已匹配前一分支。

# 复用上一例的关键字模式函数，Point 模式同样接受子类实例。
print(describe_point(point))  # 纵轴上，高度 4

纵轴高度 4
纵轴上，高度 4


## 本章小结

（1）继承沿用和重写行为，组合通过持有其他对象来协作；鸭子类型让调用方按操作约定使用不同对象。

（2）super 根据实例所属类的 MRO 继续查找，可能走向兄弟类；一致的参数约定和完整的协作调用共同决定执行链。

（3）ABC 限制尚未实现抽象方法的类实例化，但实现是否符合接口含义仍需检查。

（4）特殊方法让对象参与文字表示、长度、真值、索引、成员检测、迭代和比较；需要分别遵守返回值与回退规则。

（5）NotImplemented 表示当前方法不支持比较，按值相等的可哈希对象必须有相同哈希；可变值对象宜保持不可哈希。

（6）类模式先检查类型，再检查属性；位置模式通过 \_\_match\_args\_\_ 映射到属性名。

自查：能否解释 Left.steps 中同一行 super() 为什么会走向不同类，以及“实现了某个特殊方法”为什么不代表实现了整个容器接口？

## 练习

### 练习 1：预测 MRO 与调用结果

不先运行代码，分别写出三行输出，并说明最后一行从哪个类之后查找。使用本章定义的 Reverse、Left 和 Branch；预测完成后再运行核对，指出共同祖先为何只出现在各次协作结果中一次。

In [25]:
# 先写出 MRO 与每次 steps 的调用次序，再逐项核对三个列表。
print([base.__name__ for base in Reverse.__mro__])
print(Reverse().steps())
print(super(Left, Branch()).steps())

['Reverse', 'Right', 'Left', 'Root', 'object']
['Right', 'Left', 'Root']
['Right', 'Root']


### 练习 2：组合一个可重复遍历的队列

补全 MessageQueue：初始化时保存传入消息的列表副本，实现 \_\_len\_\_、\_\_iter\_\_ 和 \_\_contains\_\_。不继承 list，不使用 yield，不单独定义 \_\_bool\_\_。

完成后检查：空队列长度为 0 且真值为 False；用 ["早", "晚"] 创建的队列长度为 2，两次 list() 转换均保留消息次序，"早" 的成员检测为 True，"午" 为 False。

再说明：此实现用了组合，却仍可参与 len、in 和遍历，这与对象协议是什么关系？

In [26]:
class MessageQueue:
    """练习占位：在此添加初始化与所需特殊方法。"""

    pass


# 完成类后，在此添加空队列和两个消息的检查。
# 当前占位只定义类，不实例化或调用尚未实现的方法。

### 练习 3：为可变记录设计相等与模式

编写 ItemRecord，用 code 属性保存可修改的字符串编号；只有实际类型相同且编号相同才相等，对不支持的类型返回 NotImplemented。保持实例不可哈希，并声明一个位置模式对应 code 属性。

完成后检查：两个编号相同的不同实例满足 ==、不满足 is；修改其中一个编号后不再相等；类的 \_\_hash\_\_ 为 None。再用 case ItemRecord("P1") 匹配编号为 "P1" 的实例。

最后说明：为什么这里不能用 NotImplementedError 替代 NotImplemented？如果需要按当前编号建立字典索引，应选取什么作为键？

In [27]:
class ItemRecord:
    """练习占位：添加编号、相等规则和位置模式声明。"""

    pass


# 完成类后，在此添加比较、修改编号和 match 检查。
# 当前占位可以顺序执行，不制造预期外的异常。

## 参考与引用来源

| 网站 | 已核查页面、小节及支持内容 |
| --- | --- |
| docs.python.org（Python 3.12） | [类：继承与重写](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#inheritance)、[多重继承](https://docs.python.org/zh-cn/3.12/tutorial/classes.html#multiple-inheritance)；[编程 FAQ：委托](https://docs.python.org/zh-cn/3.12/faq/programming.html#what-is-delegation)，支持对象组合后的委托用法；[术语表：鸭子类型](https://docs.python.org/zh-cn/3.12/glossary.html#term-duck-typing)、[可哈希](https://docs.python.org/zh-cn/3.12/glossary.html#term-hashable)；[isinstance](https://docs.python.org/zh-cn/3.12/library/functions.html#isinstance)、[issubclass](https://docs.python.org/zh-cn/3.12/library/functions.html#issubclass)、[super](https://docs.python.org/zh-cn/3.12/library/functions.html#super)、[iter](https://docs.python.org/zh-cn/3.12/library/functions.html#iter)；[C3 方法解析顺序](https://docs.python.org/zh-cn/3.12/howto/mro.html#the-beginning)，该历史说明所述算法也用于 Python 3；[abc.ABC](https://docs.python.org/zh-cn/3.12/library/abc.html#abc.ABC)、[abc.abstractmethod](https://docs.python.org/zh-cn/3.12/library/abc.html#abc.abstractmethod)，支持实例化限制、普通实现与抽象实现；[函数定义中的装饰器](https://docs.python.org/zh-cn/3.12/reference/compound_stmts.html#function-definitions)；[f-string 转换标记 !r](https://docs.python.org/zh-cn/3.12/reference/lexical_analysis.html#f-strings)；[数据模型：文字表示](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__repr__)、[面向阅读的文字表示](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__str__)、[相等比较](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__eq__)、[哈希及自动禁用规则](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__hash__)、[真值方法](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__bool__)、[长度](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__len__)、[索引读取](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__getitem__)、[成员检测](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__contains__)、[迭代入口](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#object.__iter__)、[特殊方法查找](https://docs.python.org/zh-cn/3.12/reference/datamodel.html#special-method-lookup)；[迭代器类型](https://docs.python.org/zh-cn/3.12/library/stdtypes.html#iterator-types)，支持可迭代对象与迭代器的区分；[NotImplemented](https://docs.python.org/zh-cn/3.12/library/constants.html#NotImplemented)、[NotImplementedError](https://docs.python.org/zh-cn/3.12/library/exceptions.html#NotImplementedError)；[类模式](https://docs.python.org/zh-cn/3.12/reference/compound_stmts.html#class-patterns)，支持类型检查、属性匹配与位置转换；[typing.Protocol](https://docs.python.org/zh-cn/3.12/library/typing.html#typing.Protocol)，用于区分本章运行时约定与后续静态类型协议。 |